# Model 4 — Fast Car Classifier (Naive Bayes)

## Research Question
Based on a car's body type, fuel type, drivetrain, and transmission — can we predict whether it is a fast car (0–100 km/h under 9 seconds)?

## Introduction
This notebook applies **Gaussian Naive Bayes** to classify cars as "fast" or "slow" based on categorical and encoded features.

**Rules for this notebook:**
- Uses **unscaled data** (`proceed_dataset_without_scaling.csv`)
- You **must use `GaussianNB`** — no other algorithm is acceptable
- You **must create the `hızlı_araba` binary column** before training (see Feature Engineering)
- **Note:** Naive Bayes assumes feature independence — this is an approximation that should be acknowledged in your analysis
- You **may choose different features**, add features, and tune hyperparameters — but you **cannot change the general technique category** (must remain Naive Bayes)

## Data Import

Run this cell as-is. It loads all required libraries and the unscaled dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv('../../data/proceed_dataset_without_scaling.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## Feature Engineering

Run this cell as-is. It creates the binary target column `hızlı_araba`:
- **1** = fast car (0–100 km/h in under 9 seconds)
- **0** = slow car

The source column `Hızlanma (0-100)` is **excluded from features** in the next step to prevent data leakage.

In [ ]:
# Create binary target: 1 = fast car (0-100 km/h in under 9 seconds), 0 = slow car
# We drop 'Hızlanma (0-100)' from features after creating the target to avoid data leakage
df['hızlı_araba'] = (df['Hızlanma (0-100)'] < 9.0).astype(int)
print(f"Fast cars: {df['hızlı_araba'].sum()} ({df['hızlı_araba'].mean()*100:.1f}%)")
print(f"Slow cars: {(df['hızlı_araba'] == 0).sum()} ({(df['hızlı_araba'] == 0).mean()*100:.1f}%)")

## Feature Selection

**TODO (Student Task):** Select your features from the dataset.

- You may use the recommended list below or choose your own columns
- Do **NOT** include `Hızlanma (0-100)` — it would cause data leakage
- You may add or remove features based on your analysis

In [ ]:
# TODO: Select your features. Do NOT include 'Hızlanma (0-100)' — it would cause data leakage.
recommended_features = [
    'Kasa Tipi_Coupe', 'Kasa Tipi_SUV', 'Kasa Tipi_Crossover',
    'Çekiş_AWD (Elektronik)', 'Çekiş_Arkadan İtiş',
    'Yakıt Tipi_Hibrit', 'Yakıt Tipi_Elektrik', 'Yakıt Tipi_Dizel',
    'Vites Tipi', 'Kimden_Yetkili Bayiden', 'is_Nissan'
]
features = [f for f in recommended_features if f in df.columns]
target = 'hızlı_araba'

X = df[features].fillna(0)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Class balance in train: {y_train.value_counts().to_dict()}")

## Model Training

**TODO (Student Task):** Train the Gaussian Naive Bayes model.

- You **must** use `GaussianNB` from `sklearn.naive_bayes`
- Experiment with the `var_smoothing` parameter (e.g., `1e-9`, `1e-6`, `1e-3`)
- General structure: **instantiate → fit → predict**

In [ ]:
from sklearn.naive_bayes import GaussianNB

# GaussianNB has minimal hyperparameters — you may experiment with var_smoothing
model = GaussianNB(var_smoothing=1e-9)  # TODO: try 1e-6, 1e-3

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # probability of being a fast car
print("Naive Bayes model trained.")

## Evaluation

### Classification Report

Displays precision, recall, F1-score, and support for each class (Slow / Fast).

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace y_test and y_pred with your actual outputs after training.
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Slow (0)', 'Fast (1)']))

### Confusion Matrix

Shows the number of true positives, false positives, true negatives, and false negatives.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace y_test and y_pred with your actual outputs after training.
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Slow', 'Fast'], yticklabels=['Slow', 'Fast'],
            linewidths=1, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('Actual Label', fontsize=13)
ax.set_title('Confusion Matrix — Fast Car Classifier (Naive Bayes)', fontsize=14)
plt.tight_layout()
plt.show()

### Predicted Probability of Being a Fast Car by Body Type

For each body type present in the feature set, this bar chart shows the average predicted probability that a car of that type is a "fast car" according to the Naive Bayes model.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model and df after training.

In [ ]:
# ⚠️ Replace with your actual model and df after training.
body_type_cols = [c for c in features if 'Kasa Tipi_' in c]
if body_type_cols:
    prob_by_body = {}
    for col in body_type_cols:
        mask = df[col] == 1
        if mask.sum() > 0:
            subset = df.loc[mask, features].fillna(0)
            proba = model.predict_proba(subset)[:, 1].mean()
            prob_by_body[col.replace('Kasa Tipi_', '')] = proba

    prob_df = pd.Series(prob_by_body).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    prob_df.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
    ax.set_xlabel('Body Type', fontsize=13)
    ax.set_ylabel('Average Predicted P(Fast Car)', fontsize=13)
    ax.set_title('Average Probability of Being a Fast Car by Body Type\n(Naive Bayes)', fontsize=14)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='navy', linestyle='--', linewidth=1.5, label='50% threshold')
    ax.legend()
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("No Kasa Tipi_ columns found in features. Adjust feature selection.")

### Hand-Crafted Demo Predictions

Runs the trained model on three manually specified example cars to illustrate how predictions work for different feature combinations.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model and feature list after training.

In [ ]:
# ⚠️ Replace with your actual model and feature list after training.
# Three hand-crafted example cars for demonstration
examples = {
    'AWD + Hybrid + Automatic': {'Çekiş_AWD (Elektronik)': 1, 'Yakıt Tipi_Hibrit': 1,
                                  'Vites Tipi': 1, 'Kasa Tipi_Coupe': 0,
                                  'Kasa Tipi_SUV': 0, 'Yakıt Tipi_Elektrik': 0,
                                  'Yakıt Tipi_Dizel': 0, 'Çekiş_Arkadan İtiş': 0,
                                  'Kasa Tipi_Crossover': 0, 'Kimden_Yetkili Bayiden': 1, 'is_Nissan': 0},
    'FWD + LPG + Manual':       {'Çekiş_AWD (Elektronik)': 0, 'Yakıt Tipi_Hibrit': 0,
                                  'Vites Tipi': 0, 'Kasa Tipi_Coupe': 0,
                                  'Kasa Tipi_SUV': 0, 'Yakıt Tipi_Elektrik': 0,
                                  'Yakıt Tipi_Dizel': 0, 'Çekiş_Arkadan İtiş': 0,
                                  'Kasa Tipi_Crossover': 0, 'Kimden_Yetkili Bayiden': 0, 'is_Nissan': 0},
    'RWD + Coupe + Automatic':  {'Çekiş_AWD (Elektronik)': 0, 'Yakıt Tipi_Hibrit': 0,
                                  'Vites Tipi': 1, 'Kasa Tipi_Coupe': 1,
                                  'Kasa Tipi_SUV': 0, 'Yakıt Tipi_Elektrik': 0,
                                  'Yakıt Tipi_Dizel': 0, 'Çekiş_Arkadan İtiş': 1,
                                  'Kasa Tipi_Crossover': 0, 'Kimden_Yetkili Bayiden': 0, 'is_Nissan': 0}
}

print("=== Demo Predictions ===\n")
for name, vals in examples.items():
    # Build a row with all features the model was trained on
    row = pd.DataFrame([{f: vals.get(f, 0) for f in features}])
    prob = model.predict_proba(row)[0][1]
    pred = model.predict(row)[0]
    label = "FAST 🏎️" if pred == 1 else "SLOW 🐌"
    print(f"{name}: {label} | P(fast) = {prob*100:.1f}%")

## ⚠️ If Your Model Underperforms

If your model produces poor or surprising results (e.g., very low accuracy, unexpected associations, or trivial rules), **do not discard your results**.

- Keep all outputs as-is
- In your presentation, document exactly what you observe
- Write a short hypothesis: Why might the model have failed? (e.g., 'The dataset may not contain enough transactions with rare feature combinations to produce reliable high-confidence rules')